# DAST Retrieval Engine — Run Notebook

This notebook replaces the old top-to-bottom `run.py`. Each step is isolated so you
can run just the part you need instead of triggering the whole pipeline every time.

**How to use it:**

| Step | What it does | When to run | Needs |
|------|--------------|-------------|-------|
| 1 | Parse `document2.pdf` → DAST JSON | Once per document | fast, no LLM |
| 2 | Browse the tree to author `questions.jsonl` | Only while writing questions | **interactive** |
| 3.1 | Engine-only retrieval sanity check | Every time you tweak the engine | fast, no LLM |
| 4 | Full DAST vs RAG bake-off | Occasionally | torch/faiss + HF token |
| Summary | Pretty-print benchmark results | After Step 4 | — |

> Step 2 is an **authoring helper** — it waits for your keyboard input. That's why it
> lives in its own cell: you simply don't run it during a normal pipeline pass.

## Step 1 — Parse `document2.pdf` → DAST

Writes `benchmark/document2.dast.json`. Run once per document (or after changing the parser).

In [ ]:
import json
from parsers.pdf_parser import PDFParser

r = PDFParser().parse('benchmark/document2.pdf')
json.dump(r.to_dict(), open('benchmark/document2.dast.json', 'w'), indent=2)
print('wrote benchmark/document2.dast.json')

## Step 2 — Browse the tree to author `questions.jsonl`  *(interactive)*

Search the parsed doc for text; it prints the matching `node_id`s (which you paste into
`questions.jsonl` as `ground_truth_node_ids`).

**Run this cell only when you're authoring questions** — it waits for your input.
Verify each `ground_truth_node_id` against this output before trusting the metrics.

In [ ]:
import json

d = json.load(open('benchmark/document2.dast.json'))

def walk(n):
    yield n
    for c in n['children']:
        yield from walk(c)

term = input('search term: ').lower()
for n in walk(d):
    txt = ((n.get('title') or '') + ' ' + (n.get('text') or '')).lower()
    if term in txt:
        loc = n.get('physical_location') or {}
        label = (n.get('title') or n.get('text') or '')[:80].replace(chr(10), ' ')
        print(n['node_id'], '| p', loc.get('page_start'), '|', label)

# e.g. type: 'job title' -> gives doc.10.11.6 (p7) etc. to use as ground_truth_node_ids

## Step 3.1 — Engine-only sanity check  *(no LLM, instant)*

Runs the retriever over every question and reports how often the ground-truth node is
returned. Two views:

* **Strict** — the exact ground-truth `node_id` is in the top-k.
* **Ancestor/descendant** — a related node (parent/child on the same path) is in the top-k,
  which is a legitimate hit in hierarchical retrieval.

In [ ]:
import json
from engine.index import DASTIndex
from engine.retriever import DASTRetriever

def related(a, b):
    return a == b or b.startswith(a + '.') or a.startswith(b + '.')

idx = DASTIndex.from_json('benchmark/document2.dast.json')
eng = DASTRetriever(idx, strategy='best_first')

strict = lenient = total = 0
for line in open('benchmark/questions.jsonl'):
    line = line.strip()
    if not line:
        continue
    ex = json.loads(line)
    total += 1
    got = [x.node_id for x in eng.retrieve(ex['question'], top_k=8)]
    gt = ex['ground_truth_node_ids'][0]
    s = gt in got
    l = any(related(n, gt) for n in got)
    strict += s
    lenient += l
    print(('L-HIT' if l else 'MISS '), 'S' if s else '.', ex['qid'], '| gt', gt, '| top5', got[:5])

print(f'\nSTRICT hit@8: {strict}/{total}  |  ANCESTOR/DESCENDANT hit@8: {lenient}/{total}')

### (Optional) Spot-check a single query

Handy for eyeballing scores and matched terms on one question.

In [ ]:
for x in eng.retrieve('tips for writing job descriptions', top_k=3):
    print(round(x.score, 3), x.node_id, x.matched_terms)

## Step 4 — Full bake-off: DAST vs RAG baseline  *(heavy)*

This one needs the heavy deps and a HuggingFace token:

```bash
uv pip install -r RAG/requirements.txt \
  --index-url https://pypi.ci.artifacts.walmart.com/artifactory/api/pypi/external-pypi/simple \
  --allow-insecure-host pypi.ci.artifacts.walmart.com
brew install tesseract   # for OCR in PDFProcessor
```

Set your token/model in the cell below (or drop them in a `.env` as `RAG/README.md` describes).

In [ ]:
import os

os.environ.setdefault('HF_TOKEN', 'hf_XXXX')                              # replace with your HuggingFace token
os.environ.setdefault('DEFAULT_ANSWER_MODEL', 'Qwen/Qwen2.5-3B-Instruct')  # ungated; swap if you prefer

In [ ]:
from eval.run_benchmark import run_benchmark

result = run_benchmark(
    dast_path='benchmark/document2.dast.json',
    questions_path='benchmark/questions.jsonl',
    pdf_path='benchmark/document2.pdf',
    output_dir='results',
    top_k=8,
)
print('Benchmark complete:', result)

## Summary — pretty-print the benchmark results

In [ ]:
import json

data = json.load(open(result['output_path']))
print('=== SUMMARY ===')
print(json.dumps(data['summary'], indent=2))